# Metrics and Review Notebook

This notebook helps you review a transcription run, compute gross metrics, and perform the manual speaker relabelling step. All API calls go through the `meetings` package — no vendor API keys are used directly in this notebook.

## 1. Setup

Set the run directory and load the transcript and metadata.

In [ ]:
from pathlib import Path

from meetings.io import read_meta, read_run, read_transcript
from meetings.notebook_helpers import compute_gross_metrics

# Set your run directory here
RUN_DIR = Path("Transcription/<run_id>")

# Load transcript and metadata
transcript = read_transcript(RUN_DIR)
meta = read_meta(RUN_DIR)

# Show current stage
stage = meta.extra.get("stage", "unknown")
print(f"Current stage: {stage}")
print(f"Backend: {meta.backend}")
print(f"Duration: {meta.duration:.1f} seconds")

## 2. Gross Metrics

Compute and display gross metrics about the transcript.

In [ ]:
from rich.console import Console
from rich.table import Table

metrics = compute_gross_metrics(transcript)

console = Console()
table = Table(title="Gross Metrics")
table.add_column("Metric", style="cyan")
table.add_column("Value", style="magenta")

table.add_row("Number of speakers", str(metrics["n_speakers"]))
table.add_row("Number of segments", str(metrics["n_segments"]))
table.add_row("Number of words", str(metrics["n_words"]))
table.add_row("Duration (seconds)", f"{metrics['duration']:.1f}")

console.print(table)

In [ ]:
# Speaker distribution table
table = Table(title="Speaker Distribution")
table.add_column("Speaker", style="cyan")
table.add_column("Words", style="magenta")
table.add_column("Seconds", style="green")
table.add_column("% of Meeting", style="yellow")
table.add_column("Words/Min", style="blue")

for speaker in transcript.speakers:
    words = metrics["words_per_speaker"].get(speaker, 0)
    seconds = metrics["seconds_per_speaker"].get(speaker, 0.0)
    pct = metrics["pct_per_speaker"].get(speaker, 0.0)
    wpm = metrics["words_per_minute_per_speaker"].get(speaker, 0.0)
    table.add_row(speaker, str(words), f"{seconds:.1f}", f"{pct:.1f}%", f"{wpm:.1f}")

console.print(table)

In [ ]:
# Bar chart of speaker seconds
import matplotlib.pyplot as plt

speakers = transcript.speakers
seconds = [metrics["seconds_per_speaker"].get(s, 0.0) for s in speakers]

plt.figure(figsize=(10, 6))
plt.bar(speakers, seconds)
plt.xlabel("Speaker")
plt.ylabel("Speaking Time (seconds)")
plt.title("Speaker Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Snippet Listener / Relabel UI

Listen to speaker snippets and assign real names.

In [ ]:
from IPython.display import Audio, display

# For each speaker, find the two longest utterances
speaker_utterances = {}
for speaker in transcript.speakers:
    speaker_segments = [s for s in transcript.segments if s.speaker == speaker]
    # Sort by duration and pick the two longest
    sorted_segments = sorted(speaker_segments, key=lambda s: s.end - s.start, reverse=True)
    speaker_utterances[speaker] = sorted_segments[:2]

# Display snippets and utterances
for speaker in transcript.speakers:
    print(f"\n{'='*60}")
    print(f"Speaker: {speaker}")
    print(f"{'='*60}")
    
    # Try to find and display audio snippet
    snippet_dir = RUN_DIR / "snippets"
    snippet_files = sorted(snippet_dir.glob(f"{speaker}_*.wav")) if snippet_dir.exists() else []
    
    if snippet_files:
        for snippet_file in snippet_files[:1]:  # Play first snippet
            print(f"\nPlaying snippet: {snippet_file.name}")
            display(Audio(str(snippet_file)))
    else:
        print(f"\nNo audio snippets found for {speaker} in {snippet_dir}")
    
    # Show the two longest utterances
    utterances = speaker_utterances[speaker]
    print("\nTwo longest utterances:")
    for i, seg in enumerate(utterances, 1):
        duration = seg.end - seg.start
        print(f"  {i}. [{duration:.1f}s] {seg.text}")

### Assign Real Names

Edit the dictionary below to assign real names to each speaker label.

In [ ]:
# Edit this dictionary to assign real names
speaker_names = {}
for speaker in transcript.speakers:
    speaker_names[speaker] = None  # Replace None with the real name

# Example:
# speaker_names = {
#     "SPEAKER_00": "Alice",
#     "SPEAKER_01": "Bob",
# }

print("Speaker name mapping:")
for label, name in speaker_names.items():
    print(f"  {label} -> {name or '(not set)' }")

### Write speakers.json

Run this cell to write the mapping to `speakers.json` in the run directory.

In [ ]:
import json

speakers_path = RUN_DIR / "speakers.json"
speakers_path.write_text(
    json.dumps(speaker_names, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"Wrote speaker mapping to {speakers_path}")
print("\nNext step: Run the following command in your terminal:")
print(f"  uv run meetings relabel \"{RUN_DIR}\"")

## 4. Post-Summary Review

This section runs only if `summary.json` exists in the run directory.

In [ ]:
summary_path = RUN_DIR / "summary.json"

if summary_path.exists():
    run = read_run(RUN_DIR)
    
    # Render summary.md inline
    summary_md = (RUN_DIR / "summary.md").read_text(encoding="utf-8")
    print("=== SUMMARY.MD ===")
    print(summary_md)
    
    # List action items as a pandas DataFrame
    import pandas as pd
    
    if run.summary.action_items:
        actions_data = []
        for action in run.summary.action_items:
            actions_data.append({
                "Task": action.task,
                "Owner": action.owner or "(unassigned)",
                "Due": action.due or "(no due date)",
            })
        df = pd.DataFrame(actions_data)
        print("\n=== ACTION ITEMS ===")
        print(df.to_string(index=False))
    else:
        print("\nNo action items in summary.")
else:
    print(f"No summary.json found in {RUN_DIR}.")
    print("Run 'uv run meetings summarize <run_dir>' to generate a summary.")

## 5. Optional A/B Comparison

Compare two runs side-by-side (e.g., Claude vs Gemini, or Track A vs Track B).

In [ ]:
# Set two run directories to compare
RUN_DIR_A = Path("Transcription/<run_id_a>")
RUN_DIR_B = Path("Transcription/<run_id_b>")

if RUN_DIR_A.exists() and (RUN_DIR_A / "summary.json").exists():
    run_a = read_run(RUN_DIR_A)
    print(f"\n{'='*60}")
    print(f"RUN A: {RUN_DIR_A.name}")
    print(f"Backend: {run_a.meta.backend}")
    print(f"{'='*60}")
    summary_md_a = (RUN_DIR_A / "summary.md").read_text(encoding="utf-8")
    print(summary_md_a)
else:
    print(f"Run A not found or no summary.json: {RUN_DIR_A}")

if RUN_DIR_B.exists() and (RUN_DIR_B / "summary.json").exists():
    run_b = read_run(RUN_DIR_B)
    print(f"\n{'='*60}")
    print(f"RUN B: {RUN_DIR_B.name}")
    print(f"Backend: {run_b.meta.backend}")
    print(f"{'='*60}")
    summary_md_b = (RUN_DIR_B / "summary.md").read_text(encoding="utf-8")
    print(summary_md_b)
else:
    print(f"Run B not found or no summary.json: {RUN_DIR_B}")